# 1/10 Full Package Dataset Baseline Paper Experiments

Dataset input: `NT230/data/full_package_dataset_balanced_10000.zip` on Google Drive.

This notebook samples the same 1000-package subset as the RC-PAA 1/10 notebook: 500 benign + 500 malicious with `SEED=230`.

Goal: evaluate D2-style baseline aggregation on the same subset for metric comparison.

Outputs:
- `main_results_table.csv/json`: original LAMPS any-file baseline.
- `baseline_results_table.csv/json`: original any-file, raw max pooling, average pooling, majority voting.
- `package_scores.csv`: package-level scores/predictions.
- `baseline_wrong_predictions.csv`, `baseline_false_positives.csv`, `baseline_false_negatives.csv`.
- `summary.json`, `output_manifest.json`, `output_checklist.json`.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm matplotlib


In [ ]:
import os, shutil, sys, json, csv, math
from pathlib import Path
from collections import defaultdict, Counter

DRIVE_D1 = '/content/drive/My Drive/NT230/data/d1'
DRIVE_DATA = Path('/content/drive/My Drive/NT230/data')
DATASET_ZIP = DRIVE_DATA / 'full_package_dataset_balanced_10000.zip'
DATASET_NAME = 'full_package_dataset_balanced_10000'
DATASET_DIR = Path('/content') / DATASET_NAME
OUTPUT_DIR = DRIVE_DATA / 'results_1_10_package_baseline_paper_experiments'
RCPAA_OUTPUT_DIR = DRIVE_DATA / 'results_1_10_package_rcpaa_paper_experiments'
REPO_DIR = Path('/content/NT230')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Experiment output:', OUTPUT_DIR)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, '/content/NT230/src')

MODEL_SRC = Path(DRIVE_D1) / 'saved_models/checkpoint-best-acc/model.bin'
MODEL_DST = Path('/content/saved_models/checkpoint-best-acc/model.bin')
MODEL_DST.parent.mkdir(parents=True, exist_ok=True)
assert MODEL_SRC.exists(), f'Missing model: {MODEL_SRC}'
shutil.copy(MODEL_SRC, MODEL_DST)
print('model.bin:', round(MODEL_DST.stat().st_size / 1e6), 'MB')

assert DATASET_ZIP.exists(), f'Missing dataset zip: {DATASET_ZIP}'
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
!unzip -q "{DATASET_ZIP}" -d /content

if not DATASET_DIR.exists():
    candidates = [p for p in Path('/content').glob('full_package_dataset*') if p.is_dir()]
    assert candidates, 'Dataset unzip completed but no full_package_dataset* folder found'
    DATASET_DIR = candidates[0]

for name in ['files.jsonl', 'packages.jsonl']:
    src = DATASET_DIR / name
    dst = Path('/content') / f'full_package_{name}'
    assert src.exists(), f'Missing dataset file: {src}'
    shutil.copy(src, dst)
    print(dst, sum(1 for _ in open(dst, encoding='utf-8')), 'records')
print('dataset_dir:', DATASET_DIR)


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')


In [ ]:
# Imports and helpers
import random
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from lamps.agents.classifier import ClassifierAgent, FileClassification
from lamps.agents.extractor import ExtractedFile
from lamps.agents.verdict import VerdictAgent
from lamps.evaluation.metrics import classification_report, format_report
from lamps.utils import read_jsonl

SEED = 230
BATCH_SIZE = 256
RAW_THRESHOLD = 0.50
SAMPLE_PER_LABEL = 500
SAMPLE_TOTAL = SAMPLE_PER_LABEL * 2

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def write_csv(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if fieldnames is None:
        keys, seen = [], set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

def report_row(name, y_true, y_pred, category):
    rep = classification_report(y_true, y_pred).to_dict()
    row = {
        'category': category,
        'method': name,
        'accuracy': rep['accuracy'],
        'balanced_accuracy': rep['balanced_accuracy'],
        'precision_malicious': rep['precision'],
        'recall_malicious': rep['recall'],
        'f1_malicious': rep['f1'],
        'benign_precision': rep['benign']['precision'],
        'benign_recall': rep['benign']['recall'],
        'benign_f1': rep['benign']['f1'],
        'TN': rep['confusion']['TN'],
        'FP': rep['confusion']['FP'],
        'FN': rep['confusion']['FN'],
        'TP': rep['confusion']['TP'],
        'n_samples': rep['n_samples'],
    }
    return row, rep

def print_table(rows, title):
    print('\n' + title)
    cols = ['method', 'accuracy', 'balanced_accuracy', 'precision_malicious', 'recall_malicious', 'f1_malicious', 'TN', 'FP', 'FN', 'TP']
    df = pd.DataFrame(rows)[cols]
    display(df)
    return df


In [ ]:
# Load full package dataset and sample same 1/10 subset
ALL_FILE_RECORDS = list(read_jsonl(Path('/content/full_package_files.jsonl')))
ALL_PACKAGE_RECORDS = list(read_jsonl(Path('/content/full_package_packages.jsonl')))

rng = random.Random(SEED)
packages_by_label = defaultdict(list)
for record in ALL_PACKAGE_RECORDS:
    packages_by_label[int(record.get('label', -1))].append(record)

for label in [0, 1]:
    packages_by_label[label] = sorted(packages_by_label[label], key=lambda r: str(r['package']))
    rng.shuffle(packages_by_label[label])
    assert len(packages_by_label[label]) >= SAMPLE_PER_LABEL, f'Not enough label={label} packages: {len(packages_by_label[label])}'

sampled_packages = packages_by_label[0][:SAMPLE_PER_LABEL] + packages_by_label[1][:SAMPLE_PER_LABEL]
sampled_packages = sorted(sampled_packages, key=lambda r: (int(r.get('label', -1)), str(r['package'])))
selected_package_names = {str(r['package']) for r in sampled_packages}

package_records = sampled_packages
file_records = [r for r in ALL_FILE_RECORDS if str(r.get('package')) in selected_package_names]

package_labels = Counter(int(r.get('label', -1)) for r in package_records)
file_labels = Counter(int(r.get('target', r.get('label', -1))) for r in file_records)

sample_manifest = {
    'dataset_zip': str(DATASET_ZIP),
    'sampling': 'balanced package-level subset',
    'seed': SEED,
    'sample_per_label': SAMPLE_PER_LABEL,
    'sample_total': SAMPLE_TOTAL,
    'all_packages': len(ALL_PACKAGE_RECORDS),
    'all_files': len(ALL_FILE_RECORDS),
    'sampled_packages': len(package_records),
    'sampled_files': len(file_records),
    'package_labels': dict(package_labels),
    'file_labels': dict(file_labels),
    'selected_packages': [{'package': str(r['package']), 'label': int(r['label'])} for r in package_records],
}
write_json(OUTPUT_DIR / 'sample_manifest.json', sample_manifest)
write_csv(OUTPUT_DIR / 'sampled_packages.csv', sample_manifest['selected_packages'])

print('Sampled packages:', len(package_records))
print('Sampled files:', len(file_records))
print('Package labels:', package_labels)
print('File labels:', file_labels)


In [ ]:
# Extractor Agent ? same D2-style relevant-file filter
NOISY = {'tests', 'test', 'testing', 'docs', 'doc', 'examples', '_vendor', 'vendor'}

def is_relevant(path):
    parts = str(path).lower().replace(chr(92), '/').split('/')
    return not any(p in NOISY for p in parts) and not parts[-1].startswith('test_')

filtered = [r for r in file_records if is_relevant(r.get('path', ''))]
files = [
    ExtractedFile(
        package=str(r['package']),
        path=Path('<memory>'),
        rel_path=str(r.get('path', '')),
        source=str(r['func']),
    )
    for r in filtered
]

print('Filtered files:', len(filtered), '/', len(file_records))
print('Filtered labels diagnostic:', Counter(int(r.get('target', -1)) for r in filtered))


In [ ]:
# Classifier Agent ? CodeBERT per-file with cache
CLASSIFICATION_CACHE = OUTPUT_DIR / 'file_classifications.jsonl'
RCPAA_CLASSIFICATION_CACHE = RCPAA_OUTPUT_DIR / 'file_classifications.jsonl'

if CLASSIFICATION_CACHE.exists():
    print('Loading cached classifications:', CLASSIFICATION_CACHE)
    classification_rows = list(read_jsonl(CLASSIFICATION_CACHE))
elif RCPAA_CLASSIFICATION_CACHE.exists():
    print('Reusing RC-PAA 1/10 cached classifications:', RCPAA_CLASSIFICATION_CACHE)
    classification_rows = list(read_jsonl(RCPAA_CLASSIFICATION_CACHE))
    write_jsonl(CLASSIFICATION_CACHE, classification_rows)
else:
    classifier = ClassifierAgent(checkpoint=MODEL_DST, device='cuda' if torch.cuda.is_available() else 'cpu', batch_size=BATCH_SIZE)
    classifications_tmp = []
    for start in tqdm(range(0, len(files), BATCH_SIZE), desc='CodeBERT batches'):
        batch = files[start:start + BATCH_SIZE]
        classifications_tmp.extend(classifier.classify_files(batch))
    classification_rows = [
        {'package': c.package, 'rel_path': c.rel_path, 'label': c.label, 'target': int(c.target), 'score': float(c.score)}
        for c in classifications_tmp
    ]
    write_jsonl(CLASSIFICATION_CACHE, classification_rows)
    print('Saved cache:', CLASSIFICATION_CACHE)

classifications = [
    FileClassification(
        package=str(r['package']),
        rel_path=str(r['rel_path']),
        label=str(r['label']),
        target=int(r['target']),
        score=float(r['score']),
    )
    for r in classification_rows
]

assert len(classifications) == len(files), (len(classifications), len(files))
print('Classifications:', len(classifications))
print('Predicted file labels diagnostic:', Counter(int(c.target) for c in classifications))


In [ ]:
# Build package index
cls_by_pkg = defaultdict(list)
files_by_pkg = defaultdict(list)
for cls, extracted in zip(classifications, files):
    cls_by_pkg[cls.package].append(cls)
    files_by_pkg[extracted.package].append(extracted)

packages = [str(p['package']) for p in package_records]
y_true = [int(p['label']) for p in package_records]
print('Packages with classifications:', sum(1 for p in packages if cls_by_pkg.get(p)))
print('Total packages:', len(packages))


In [ ]:
# D2-style baseline aggregation and simple pooling baselines
verdict_agent = VerdictAgent(llm=None)

def predict_original_any_file(preds):
    return 1 if any(int(p.target) == 1 for p in preds) else 0

def predict_raw_max(preds, threshold=RAW_THRESHOLD):
    return 1 if max([float(p.score) for p in preds], default=0.0) >= threshold else 0

def predict_average_pooling(preds, threshold=RAW_THRESHOLD):
    if not preds:
        return 0
    return 1 if (sum(float(p.score) for p in preds) / len(preds)) >= threshold else 0

def predict_majority_voting(preds):
    if not preds:
        return 0
    return 1 if (sum(int(p.target) for p in preds) / len(preds)) > 0.5 else 0

predictions_by_method = {
    'original_any_file': [],
    'raw_max_pooling_0.50': [],
    'average_pooling_0.50': [],
    'majority_voting': [],
}
package_score_rows = []

for pkg in package_records:
    name = str(pkg['package'])
    preds = cls_by_pkg.get(name, [])
    verdict = verdict_agent.aggregate(name, preds)
    raw_max = max([float(p.score) for p in preds], default=0.0)
    average = (sum(float(p.score) for p in preds) / len(preds)) if preds else 0.0
    majority_ratio = (sum(int(p.target) for p in preds) / len(preds)) if preds else 0.0

    predictions_by_method['original_any_file'].append(int(verdict.target))
    predictions_by_method['raw_max_pooling_0.50'].append(1 if raw_max >= RAW_THRESHOLD else 0)
    predictions_by_method['average_pooling_0.50'].append(1 if average >= RAW_THRESHOLD else 0)
    predictions_by_method['majority_voting'].append(1 if majority_ratio > 0.5 else 0)

    package_score_rows.append({
        'package': name,
        'target': int(pkg['label']),
        'n_files_total': int(pkg.get('n_files', 0)),
        'n_files_after_filter': len(preds),
        'raw_max_score': raw_max,
        'average_score': average,
        'majority_ratio': majority_ratio,
        'baseline_prediction': int(verdict.target),
        'baseline_malicious_files': len(verdict.malicious_files),
        'baseline_rationale': verdict.rationale,
        'flagged_files': json.dumps([{'path': f.rel_path, 'score': f.score} for f in verdict.malicious_files[:10]], ensure_ascii=False),
    })

baseline_rows = []
baseline_reports = {}
for method, pred in predictions_by_method.items():
    row, rep = report_row(method, y_true, pred, 'baseline')
    baseline_rows.append(row)
    baseline_reports[method] = rep

main_results_rows = [r for r in baseline_rows if r['method'] == 'original_any_file']
print_table(baseline_rows, 'Baseline comparison on same 1/10 package subset')
write_csv(OUTPUT_DIR / 'main_results_table.csv', main_results_rows)
write_json(OUTPUT_DIR / 'main_results_table.json', main_results_rows)
write_csv(OUTPUT_DIR / 'baseline_results_table.csv', baseline_rows)
write_json(OUTPUT_DIR / 'baseline_results_table.json', {'rows': baseline_rows, 'reports': baseline_reports})
write_csv(OUTPUT_DIR / 'package_scores.csv', package_score_rows)
write_jsonl(OUTPUT_DIR / 'package_scores.jsonl', package_score_rows)


In [ ]:
# Threshold sweep for raw max and average pooling
thresholds = [round(x / 100, 2) for x in range(5, 96, 5)]
score_maps = {
    'raw_max_pooling': [row['raw_max_score'] for row in package_score_rows],
    'average_pooling': [row['average_score'] for row in package_score_rows],
}
threshold_rows = []
best_by_f1 = {}
for score_name, scores in score_maps.items():
    rows = []
    for th in thresholds:
        pred = [1 if float(s) >= th else 0 for s in scores]
        row, rep = report_row(f'{score_name}@{th:.2f}', y_true, pred, 'threshold_sweep')
        row['score_name'] = score_name
        row['threshold'] = th
        rows.append(row)
        threshold_rows.append(row)
    best_by_f1[score_name] = max(rows, key=lambda r: (r['f1_malicious'], r['balanced_accuracy'], r['precision_malicious']))

print('Best threshold by F1:')
display(pd.DataFrame(best_by_f1.values()))
write_csv(OUTPUT_DIR / 'threshold_sweep.csv', threshold_rows)
write_jsonl(OUTPUT_DIR / 'threshold_sweep.jsonl', threshold_rows)
write_json(OUTPUT_DIR / 'best_thresholds.json', {'by_f1': best_by_f1})


In [ ]:
# Error analysis and audit samples for baseline
baseline_pred = predictions_by_method['original_any_file']
wrong_rows, fp_rows, fn_rows = [], [], []
for row, yt, yp in zip(package_score_rows, y_true, baseline_pred):
    item = dict(row)
    item['predicted'] = int(yp)
    item['error_type'] = 'correct' if int(yt) == int(yp) else ('FP' if int(yt) == 0 else 'FN')
    if item['error_type'] != 'correct':
        wrong_rows.append(item)
        if item['error_type'] == 'FP':
            fp_rows.append(item)
        else:
            fn_rows.append(item)

wrong_rows_sorted = sorted(wrong_rows, key=lambda r: (r['error_type'], -float(r['raw_max_score']), r['package']))
fp_rows_sorted = sorted(fp_rows, key=lambda r: (-float(r['raw_max_score']), r['package']))
fn_rows_sorted = sorted(fn_rows, key=lambda r: (-float(r['raw_max_score']), r['package']))

audit_samples = {
    'false_positive_example': fp_rows_sorted[0] if fp_rows_sorted else None,
    'false_negative_example': fn_rows_sorted[0] if fn_rows_sorted else None,
    'counts': {
        'wrong': len(wrong_rows_sorted),
        'false_positives': len(fp_rows_sorted),
        'false_negatives': len(fn_rows_sorted),
    },
}

fields = ['package', 'target', 'predicted', 'error_type', 'n_files_total', 'n_files_after_filter', 'raw_max_score', 'average_score', 'majority_ratio', 'baseline_prediction', 'baseline_malicious_files', 'flagged_files']
write_csv(OUTPUT_DIR / 'baseline_wrong_predictions.csv', wrong_rows_sorted, fields)
write_csv(OUTPUT_DIR / 'baseline_false_positives.csv', fp_rows_sorted, fields)
write_csv(OUTPUT_DIR / 'baseline_false_negatives.csv', fn_rows_sorted, fields)
write_jsonl(OUTPUT_DIR / 'baseline_wrong_predictions.jsonl', wrong_rows_sorted)
write_jsonl(OUTPUT_DIR / 'baseline_false_positives.jsonl', fp_rows_sorted)
write_jsonl(OUTPUT_DIR / 'baseline_false_negatives.jsonl', fn_rows_sorted)
write_json(OUTPUT_DIR / 'audit_samples.json', audit_samples)

print('Wrong/FP/FN:', len(wrong_rows_sorted), len(fp_rows_sorted), len(fn_rows_sorted))
display(pd.DataFrame(wrong_rows_sorted)[fields].head(10) if wrong_rows_sorted else pd.DataFrame())


In [ ]:
# Summary and figures
output_manifest = {
    'main_results': ['main_results_table.csv', 'main_results_table.json'],
    'baseline_results': ['baseline_results_table.csv', 'baseline_results_table.json'],
    'threshold_sensitivity': ['threshold_sweep.csv', 'threshold_sweep.jsonl', 'best_thresholds.json'],
    'audit_and_errors': ['audit_samples.json', 'baseline_wrong_predictions.csv', 'baseline_false_positives.csv', 'baseline_false_negatives.csv'],
    'sample': ['sample_manifest.json', 'sampled_packages.csv'],
}
output_checklist = {
    group: {'complete': all((OUTPUT_DIR / name).exists() for name in files), 'files': files}
    for group, files in output_manifest.items()
}
write_json(OUTPUT_DIR / 'output_manifest.json', output_manifest)
write_json(OUTPUT_DIR / 'output_checklist.json', output_checklist)
write_json(OUTPUT_DIR / 'summary.json', {
    'n_packages': len(package_records),
    'n_files_before_filter': len(file_records),
    'n_filtered_files': len(filtered),
    'package_labels': dict(package_labels),
    'raw_threshold_fixed': RAW_THRESHOLD,
    'sampling': sample_manifest,
    'baseline_results': baseline_rows,
    'best_thresholds': {'by_f1': best_by_f1},
    'audit_counts': audit_samples['counts'],
    'output_checklist': output_checklist,
    'output': str(OUTPUT_DIR),
})

main_df = pd.DataFrame(baseline_rows).set_index('method').loc[list(predictions_by_method)].reset_index()
plt.figure(figsize=(9, 5))
x = range(len(main_df))
plt.bar([i - 0.18 for i in x], main_df['f1_malicious'], width=0.36, label='F1 malicious')
plt.bar([i + 0.18 for i in x], main_df['balanced_accuracy'], width=0.36, label='Balanced accuracy')
plt.xticks(list(x), main_df['method'], rotation=25, ha='right')
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.title('D2-style baseline aggregation on 1/10 package subset')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figure_1_baseline_results.png', dpi=200)
plt.show()

plt.figure(figsize=(9, 5))
plt.bar([i - 0.18 for i in x], main_df['FP'], width=0.36, label='False Positive')
plt.bar([i + 0.18 for i in x], main_df['FN'], width=0.36, label='False Negative')
plt.xticks(list(x), main_df['method'], rotation=25, ha='right')
plt.ylabel('Package count')
plt.title('Baseline error breakdown')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figure_2_baseline_fp_fn.png', dpi=200)
plt.show()

print('Checklist:')
display(pd.DataFrame([{'group': k, **v} for k, v in output_checklist.items()]))
print('Saved outputs:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(p.name)
